In [ ]:
# !pip install transformers torch datasets huggingface_hub pillow opencv-python ipywidgets

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import torch

# GPU checks
print("Python:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("cuDNN enabled:", torch.backends.cudnn.enabled)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Evaluator (dh batee2, esta5demo el ta7t)

In [ ]:
import torch
from transformers import pipeline, AutoImageProcessor

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

processor = AutoImageProcessor.from_pretrained(
    "buildborderless/CommunityForensics-DeepfakeDet-ViT"
)
processor.size = {"height": 384, "width": 384}

img_pipeline = pipeline(
    "image-classification",
    model="buildborderless/CommunityForensics-DeepfakeDet-ViT",
    image_processor=processor,
    device=0 if torch.cuda.is_available() else -1,
)



In [ ]:
print(img_pipeline.model.config.id2label)
print(img_pipeline.model.config.label2id)

label_map = {       # n8ayaro 3la asas el model, mape el real le 0 wl fake le 1
    "LABEL_0": 0,
    "LABEL_1": 1,
}

In [ ]:
from evaluator import evaluate_all_datasets
results = evaluate_all_datasets(
    pipeline_obj=img_pipeline,
    model_name="CommunityForensics-DeepfakeDet-ViT_TEST",
    batch_size=16,
    checkpoint_every=50,    # saves kol 50 batch
    # max_batches=3,        # uncomment lw hn-test 7aga bs 3la kam batch kda
    label_map=label_map,
)

# Faster Evaluator

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

model_name = "buildborderless/CommunityForensics-DeepfakeDet-ViT"

processor = AutoImageProcessor.from_pretrained(model_name)
processor.size = {"height": 384, "width": 384}

model = AutoModelForImageClassification.from_pretrained(model_name)


In [ ]:

print(model.config.id2label)
print(model.config.label2id)

label_map = {   # n8ayaro 3la asas el model, mape el real le 0 wl fake le 1
    "LABEL_0": 0,  # real
    "LABEL_1": 1,  # fake
}

In [ ]:
from faster_evaluator import evaluate_all_datasets_fast

results = evaluate_all_datasets_fast(
    model=model,
    processor=processor,
    model_name="CommunityForensics-DeepfakeDet-ViT_TEST",
    batch_size=16,
    label_map=label_map,
    checkpoint_every=100,
    num_workers=4,
    # max_batches=3,
)

# Model 2

In [1]:
from pathlib import Path
import sys

REPO_DIR = Path("M2F2_Det").resolve()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(REPO_DIR.exists())
print((REPO_DIR / "checkpoints" / "stage_1").exists())
print((REPO_DIR / "utils" / "weights").exists())

True
True
True


In [2]:
%load_ext autoreload
%autoreload 2

from m2f2_fast_evaluator import (
    load_m2f2_stage1,
    get_m2f2_transform,
    evaluate_all_datasets_m2f2_fast,
)

model, device = load_m2f2_stage1()
transform = get_m2f2_transform()

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0..

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.embeddings.position_embedding.weight              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias 

Freezing CLIP vision encoder.
Loaded vision tower.


In [ ]:
results = evaluate_all_datasets_m2f2_fast(
    model=model,
    device=device,
    transform=transform,
    model_name="CHELSEA234_M2F2_Det",
    batch_size=8,
    checkpoint_every=100,
    num_workers=0,
    max_batches=30,
    threshold=0.5,
)


[1/4] Running dataset: 20K_real_and_deepfake_images
[1/4] Running 20K_real_and_deepfake_images: 17354 remaining


[1/4] 20K_real_and_deepfake_images:   0%|          | 0/2 [00:00<?, ?it/s]


[2/4] Running dataset: DeepDetect-2025
[2/4] Running DeepDetect-2025: 112185 remaining


[2/4] DeepDetect-2025:   0%|          | 0/2 [00:00<?, ?it/s]


[3/4] Running dataset: Deepfake-vs-Real-v2
[3/4] Running Deepfake-vs-Real-v2: 19291 remaining


[3/4] Deepfake-vs-Real-v2:   0%|          | 0/2 [00:00<?, ?it/s]


[4/4] Running dataset: gravex200k
